# Transformers desde cero con PyTorch

## Implementación pedagógica de un Transformer codificador-decodificador

Este cuaderno construye un **Transformer secuencia a secuencia** con PyTorch y muestra sus componentes esenciales: codificación posicional, atención de producto escalar escalada, atención multi-cabeza, conexiones residuales, normalización de capas, redes de avance y atención cruzada entre codificador y decodificador.

> **Objetivo:** comprender la mecánica de la arquitectura. No pretende sustituir las implementaciones optimizadas de `torch.nn.Transformer` o Hugging Face para entrenamiento de producción.


## 1. Preparación del entorno

Se importan PyTorch y los módulos necesarios. También se fija una semilla para que los resultados aleatorios sean reproducibles durante la ejecución del ejemplo.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibilidad para el ejemplo pedagógico.
torch.manual_seed(42)

# Usa GPU si está disponible; el código funciona igualmente en CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo seleccionado: {device}")


## 2. Hiperparámetros

Estos valores son deliberadamente moderados para facilitar la ejecución en Jupyter o Google Colab. En el Transformer original, la configuración base usaba una dimensión del modelo de 512, ocho cabezas de atención y seis capas; aquí se utiliza una versión más ligera.

- `d_model`: tamaño de la representación interna de cada token.
- `num_heads`: número de perspectivas paralelas en la atención multi-cabeza.
- `d_ff`: tamaño interno de la red de avance por posición.
- `num_layers`: profundidad tanto del codificador como del decodificador.


In [ ]:
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 4
D_FF = 1024
DROPOUT = 0.1
MAX_SEQ_LENGTH = 128

SRC_VOCAB_SIZE = 10_000
TGT_VOCAB_SIZE = 10_000
PAD_IDX = 0

assert D_MODEL % NUM_HEADS == 0, "D_MODEL debe ser divisible por NUM_HEADS."
print(f"Dimensión por cabeza: {D_MODEL // NUM_HEADS}")


## 3. Codificación posicional

La autoatención no incorpora el orden de los tokens por sí sola. Por ello, se añade una señal posicional a los embeddings. La versión siguiente usa codificaciones sinusoidales, que no añaden parámetros entrenables y se almacenan como un *buffer* del modelo.

El método `register_buffer` asegura que el tensor se mueva junto al modelo entre CPU y GPU, pero no se actualice mediante optimización.


In [ ]:
class PositionalEncoding(nn.Module):
    """Añade información posicional sinusoidal a embeddings de forma (lote, longitud, d_model)."""

    def __init__(self, d_model, max_seq_length, dropout=0.1):
        super().__init__()

        positions = torch.arange(max_seq_length).unsqueeze(1).float()
        frequencies = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10_000.0) / d_model)
        )

        pe = torch.zeros(max_seq_length, d_model)
        pe[:, 0::2] = torch.sin(positions * frequencies)
        pe[:, 1::2] = torch.cos(positions * frequencies)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Selecciona solo las posiciones necesarias para la longitud de la secuencia actual.
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


## 4. Atención de producto escalar escalada

Este es el núcleo matemático de la atención. Las consultas (`q`) se comparan con las claves (`k`) para producir puntajes de relevancia; después, una distribución normalizada pondera los valores (`v`).

La máscara permite dos comportamientos esenciales:

- Ignorar posiciones de relleno (*padding*).
- Impedir que el decodificador acceda a tokens futuros durante la generación autorregresiva.


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    """
    Calcula atención escalada.

    q, k, v tienen forma: (lote, cabezas, longitud, dimensión_por_cabeza).
    mask debe ser booleana y permitir difusión hacia la matriz de puntajes.
    """
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # False marca posiciones que el modelo no puede atender.
        scores = scores.masked_fill(~mask, float("-inf"))

    attention_weights = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    output = attention_weights @ v
    return output, attention_weights


## 5. Atención multi-cabeza

La atención multi-cabeza proyecta las representaciones de entrada en varios subespacios. Cada cabeza calcula atención de manera independiente; posteriormente, las salidas se combinan y se proyectan de vuelta a la dimensión del modelo.

La opción `return_attention=True` permite recuperar los pesos de atención para análisis o visualización.


In [ ]:
class MultiHeadAttention(nn.Module):
    """Implementación educativa de atención multi-cabeza."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads."

        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.shape
        x = x.view(batch_size, seq_length, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_length, self.num_heads * self.d_k)

    def forward(self, query, key, value, mask=None, return_attention=False):
        q = self.split_heads(self.w_q(query))
        k = self.split_heads(self.w_k(key))
        v = self.split_heads(self.w_v(value))

        attended, attention_weights = scaled_dot_product_attention(
            q, k, v, mask=mask, dropout=self.dropout
        )

        output = self.w_o(self.combine_heads(attended))

        if return_attention:
            return output, attention_weights
        return output


## 6. Red de avance por posición

Después de mezclar contexto con atención, cada token pasa por la misma red no lineal. Esta capa transforma las representaciones de manera independiente por posición y complementa las relaciones entre tokens aprendidas por la atención.

Se utiliza GELU, una activación frecuente en Transformers modernos.


In [ ]:
class FeedForward(nn.Module):
    """Red no lineal aplicada de forma independiente a cada posición."""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


## 7. Bloque del codificador

Cada capa del codificador combina dos subcapas:

1. Autoatención no causal sobre la secuencia de entrada.
2. Red de avance por posición.

La implementación utiliza normalización previa (*pre-norm*), una configuración frecuente en modelos profundos modernos. Cada subcapa conserva una conexión residual que facilita el flujo de información durante el entrenamiento.


In [ ]:
class EncoderLayer(nn.Module):
    """Una capa de codificador con autoatención, pre-norm y conexiones residuales."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        normalized_x = self.norm1(x)
        x = x + self.dropout(
            self.self_attention(normalized_x, normalized_x, normalized_x, src_mask)
        )

        normalized_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(normalized_x))
        return x


## 8. Bloque del decodificador

El decodificador incorpora tres subcapas:

1. Autoatención causal en la secuencia objetivo.
2. Atención cruzada sobre la salida del codificador.
3. Red de avance por posición.

La atención causal evita que una posición consulte tokens posteriores. La atención cruzada permite que el decodificador consulte la representación contextual de la entrada.


In [ ]:
class DecoderLayer(nn.Module):
    """Una capa de decodificador con atención causal y atención cruzada."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.masked_self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        normalized_x = self.norm1(x)
        x = x + self.dropout(
            self.masked_self_attention(normalized_x, normalized_x, normalized_x, tgt_mask)
        )

        normalized_x = self.norm2(x)
        x = x + self.dropout(
            self.cross_attention(normalized_x, encoder_output, encoder_output, src_mask)
        )

        normalized_x = self.norm3(x)
        x = x + self.dropout(self.feed_forward(normalized_x))
        return x


## 9. Modelo Transformer completo

Esta clase integra embeddings, codificación posicional, pilas de codificador y decodificador, máscaras de *padding*, máscara causal y proyección final hacia el vocabulario objetivo.

El resultado tiene forma `(lote, longitud_objetivo, tamaño_vocabulario_objetivo)` y contiene **logits**, no probabilidades. Durante el entrenamiento, `CrossEntropyLoss` recibe directamente estos logits.


In [ ]:
class Transformer(nn.Module):
    """Transformer codificador-decodificador para fines didácticos."""

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        d_ff=D_FF,
        max_seq_length=MAX_SEQ_LENGTH,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length, dropout)

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model

    @staticmethod
    def create_padding_mask(tokens, pad_idx=PAD_IDX):
        # Forma: (lote, 1, 1, longitud). True indica token válido.
        return (tokens != pad_idx).unsqueeze(1).unsqueeze(2)

    @staticmethod
    def create_causal_mask(sequence_length, device):
        # Forma: (1, 1, longitud, longitud). Solo permite mirar posiciones actuales y previas.
        return torch.tril(
            torch.ones(sequence_length, sequence_length, device=device, dtype=torch.bool)
        ).unsqueeze(0).unsqueeze(0)

    def forward(self, src, tgt_input, pad_idx=PAD_IDX):
        src_mask = self.create_padding_mask(src, pad_idx)

        tgt_padding_mask = self.create_padding_mask(tgt_input, pad_idx)
        causal_mask = self.create_causal_mask(tgt_input.size(1), tgt_input.device)
        tgt_mask = tgt_padding_mask & causal_mask

        src_embeddings = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_embeddings = self.tgt_embedding(tgt_input) * math.sqrt(self.d_model)

        encoder_output = self.positional_encoding(src_embeddings)
        decoder_output = self.positional_encoding(tgt_embeddings)

        for layer in self.encoder_layers:
            encoder_output = layer(encoder_output, src_mask)

        for layer in self.decoder_layers:
            decoder_output = layer(
                decoder_output,
                encoder_output,
                tgt_mask=tgt_mask,
                src_mask=src_mask,
            )

        return self.output_projection(decoder_output)


## 10. Verificación del pase hacia adelante

Se instancia el modelo y se crean secuencias sintéticas. En un problema real, `src` y `tgt` provendrían de un tokenizador y de un conjunto de datos de pares entrada-salida, por ejemplo, una frase de origen y su traducción.

Para el entrenamiento autoregresivo se usa *teacher forcing*: el decodificador recibe la secuencia objetivo desplazada una posición a la derecha y aprende a predecir el siguiente token.


In [ ]:
model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
).to(device)

batch_size = 8
src_length = 20
tgt_length = 18

src = torch.randint(1, SRC_VOCAB_SIZE, (batch_size, src_length), device=device)
tgt = torch.randint(1, TGT_VOCAB_SIZE, (batch_size, tgt_length), device=device)

# Entrada y objetivo desplazados para entrenamiento autoregresivo.
tgt_input = tgt[:, :-1]
tgt_expected = tgt[:, 1:]

logits = model(src, tgt_input)

print(f"Forma de src: {tuple(src.shape)}")
print(f"Forma de tgt_input: {tuple(tgt_input.shape)}")
print(f"Forma de logits: {tuple(logits.shape)}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")


## 11. Pérdida y paso de optimización

La entropía cruzada compara los logits del modelo con el token objetivo esperado. Se usa `ignore_index=PAD_IDX` para que los tokens de relleno no contribuyan a la pérdida.

Este bloque ejecuta un solo paso de entrenamiento demostrativo. En un entrenamiento real se repetiría sobre lotes de datos, se monitorizarían métricas de validación y se utilizaría una planificación de la tasa de aprendizaje.


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

model.train()
optimizer.zero_grad()

logits = model(src, tgt_input)
loss = criterion(
    logits.reshape(-1, TGT_VOCAB_SIZE),
    tgt_expected.reshape(-1),
)

loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print(f"Pérdida inicial después de un paso: {loss.item():.4f}")


## 12. Comprobación de la máscara causal

La máscara causal impide que el decodificador consulte tokens posteriores. En la matriz resultante, `True` indica una posición permitida y `False` una posición bloqueada.

Esta restricción es la que hace posible entrenar en paralelo, pero generar de forma autorregresiva en inferencia.


In [ ]:
causal_mask = model.create_causal_mask(sequence_length=6, device=device)
print(causal_mask[0, 0].int().cpu())


## 13. Próximos pasos

Para convertir este ejemplo en un proyecto más cercano a producción, puedes:

- Sustituir los datos sintéticos por un corpus paralelo o una tarea de resumen.
- Usar un tokenizador real y tokens especiales como inicio, fin, relleno y desconocido.
- Separar los conjuntos de entrenamiento, validación y prueba.
- Incorporar una planificación de tasa de aprendizaje con calentamiento.
- Implementar decodificación voraz, búsqueda en haz o muestreo durante inferencia.
- Comparar esta implementación pedagógica con `torch.nn.Transformer` y Hugging Face Transformers.
- Medir pérdida, perplejidad, latencia, memoria y calidad de salida.

La finalidad del cuaderno no es reproducir un modelo de frontera, sino convertir los siete mecanismos del Transformer en componentes concretos, inspeccionables y modificables.
